# CUDA Auto-Runner

This notebook is a persistent GPU worker. It polls your GitHub repo for new `.cu` jobs,
compiles and runs them, then pushes results back.

**One-time setup:**
1. Open **Edit → Notebook settings** → set Hardware accelerator to **GPU**.
2. Open the **🔑 Secrets** sidebar and add:
   - `GH_PAT` — a GitHub Personal Access Token with `repo` scope
   - `GITHUB_REPO` — your repo, e.g. `alice/cuda-kernels`
3. Click **Runtime → Run all**.
4. The notebook will keep running until the Colab session times out (~12 h free tier).


In [ ]:
# Cell 1 — install deps (runs once)
!pip install requests -q

In [ ]:
# Cell 2 — configuration
import subprocess

# Read secrets from Colab's Secrets sidebar (recommended)
try:
    from google.colab import userdata
    GH_PAT       = userdata.get('GH_PAT')
    GITHUB_REPO  = userdata.get('GITHUB_REPO')  # e.g. 'alice/cuda-kernels'
except Exception:
    # Fallback: hardcode for local Jupyter testing
    GH_PAT      = 'ghp_YOUR_TOKEN_HERE'
    GITHUB_REPO = 'your-username/cuda-kernels'

POLL_INTERVAL = 10  # seconds between GitHub polls

# Confirm GPU
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
GPU_INFO = result.stdout.strip() if result.returncode == 0 else 'unknown GPU'
print(f'GPU: {GPU_INFO}')
print(f'Repo: {GITHUB_REPO}')
print('Configuration OK.')

In [ ]:
# Cell 3 — GitHub API helpers
import base64
import requests


def _headers():
    return {
        'Authorization': f'token {GH_PAT}',
        'Accept': 'application/vnd.github.v3+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }


def gh_get_file(path):
    """Return (content_str, sha) or (None, None) if file not found."""
    resp = requests.get(
        f'https://api.github.com/repos/{GITHUB_REPO}/contents/{path}',
        headers=_headers(), timeout=20,
    )
    if resp.status_code == 404:
        return None, None
    resp.raise_for_status()
    data = resp.json()
    content = base64.b64decode(data['content']).decode()
    return content, data['sha']


def gh_put_file(path, content, message, sha=None):
    """Create or update a file in the repo."""
    payload = {
        'message': message,
        'content': base64.b64encode(content.encode()).decode(),
    }
    if sha:
        payload['sha'] = sha
    resp = requests.put(
        f'https://api.github.com/repos/{GITHUB_REPO}/contents/{path}',
        headers=_headers(), json=payload, timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


print('GitHub helpers ready.')

In [ ]:
# Cell 4 — CUDA compile-and-run helper
import subprocess
from pathlib import Path


def run_cuda_kernel(cu_file: str) -> str:
    """
    Fetch cu_file from GitHub, compile with nvcc, run, return captured output.
    cu_file is repo-relative, e.g. 'kernels/matmul_naive.cu'.
    """
    content, _ = gh_get_file(cu_file)
    if content is None:
        return f'ERROR: file not found in repo: {cu_file}'

    filename = Path(cu_file).name
    stem     = Path(cu_file).stem
    Path(filename).write_text(content)

    # Compile
    compile_proc = subprocess.run(
        ['nvcc', '-arch=sm_75', '-O2', '-o', stem, filename],
        capture_output=True, text=True, timeout=120,
    )
    if compile_proc.returncode != 0:
        return f'COMPILATION FAILED:\n{compile_proc.stderr.strip()}'

    # Run
    run_proc = subprocess.run(
        [f'./{stem}'],
        capture_output=True, text=True, timeout=120,
    )
    output = run_proc.stdout.strip()
    if run_proc.returncode != 0:
        output += f'\nRUNTIME ERROR:\n{run_proc.stderr.strip()}'
    return output


print('Executor ready.')

In [ ]:
# Cell 5 — Main polling loop  (runs until you interrupt or session ends)
import json
import time
from datetime import datetime, timezone

processed = set()

print(f'[{datetime.now().isoformat()}] Colab CUDA executor started.')
print(f'GPU  : {GPU_INFO}')
print(f'Repo : {GITHUB_REPO}')
print(f'Poll : every {POLL_INTERVAL}s')
print('Waiting for jobs from jobs/current.json ...\n')

while True:
    try:
        content, _ = gh_get_file('jobs/current.json')

        if content:
            job = json.loads(content)
            job_sha = job.get('sha', '')
            cu_file = job.get('cu_file', '')

            if job_sha and cu_file and job.get('status') == 'pending' and job_sha not in processed:
                ts = datetime.now().isoformat(timespec='seconds')
                print(f'[{ts}] JOB  sha={job_sha[:8]}  file={cu_file}')

                # Run the kernel
                output = run_cuda_kernel(cu_file)

                # Assemble results blob
                results = '\n'.join([
                    f'GPU:       {GPU_INFO}',
                    f'Kernel:    {cu_file}',
                    f'Commit:    {job_sha[:8]}',
                    f'Timestamp: {datetime.now(timezone.utc).isoformat()}',
                    '',
                    output,
                ])

                # Push results back
                results_path = f'results/{job_sha}.txt'
                existing, existing_sha = gh_get_file(results_path)
                gh_put_file(
                    results_path,
                    results,
                    f'[ci skip] results: {cu_file} ({job_sha[:8]})',
                    existing_sha,
                )

                processed.add(job_sha)
                print(f'[{ts}] DONE results pushed → {results_path}')
                print(f'\nOutput:\n{output}\n{"-"*60}\n')

        time.sleep(POLL_INTERVAL)

    except KeyboardInterrupt:
        print('\nStopped by user.')
        break
    except Exception as exc:
        print(f'[error] {exc}  — retrying in 30s')
        time.sleep(30)
